# PitchCheck audio service on Kaggle

Runs `services/audio` (FastAPI + Gemma) on a Kaggle GPU and exposes it through a Cloudflare quick tunnel, so the web app can run on a laptop without a GPU.

Before running:

1. **Settings → Accelerator → GPU T4 x2**, and **Settings → Internet → On** (needs a phone-verified account).
2. **Add-ons → Secrets**: add `HF_TOKEN`, a Hugging Face token whose account has accepted the Gemma license.
3. Run all cells. The tunnel cell prints the URL, and the last cell keeps running. On the laptop:

```bash
AUDIO_SERVICE_URL=https://<printed-url> pnpm dev
```

The last cell follows `BRANCH`: about every 30 seconds it fetches the branch, and when a push changed something under `services/audio` it restarts the service on the new code. It does not fetch while an analysis is running. The tunnel keeps running, so the URL stays the same; the service does not answer while Gemma loads again.

Leave that cell running. Interrupting it has not been tried on Kaggle: in Jupyter an interrupt can also reach the processes a notebook started, which here are the service and the tunnel.

The URL changes every time the tunnel cell runs, and the session stops when the notebook is idle or after about 12 hours.

In [ ]:
# audio input: google/gemma-4-E2B-it | google/gemma-4-E4B-it | google/gemma-4-12B-it
GEMMA_MODEL = "google/gemma-4-E2B-it"
BRANCH = "main"

In [ ]:
%cd /kaggle/working
!rm -rf pitchcheck && git clone -q --depth 1 -b {BRANCH} https://github.com/tomovicj/pitchcheck.git
%cd /kaggle/working/pitchcheck/services/audio

# Kaggle already has CUDA builds of torch and torchvision; requirements.txt would pull CPU wheels
!grep -vE '^(--extra-index-url|torch|torchvision)\b' requirements.txt > /tmp/requirements-kaggle.txt
!pip install -q -r /tmp/requirements-kaggle.txt
!which ffmpeg > /dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null)

In [ ]:
import os, subprocess, time, urllib.request
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator to a GPU"
env = {
    **os.environ,
    "HF_TOKEN": UserSecretsClient().get_secret("HF_TOKEN"),
    "GEMMA_MODEL": GEMMA_MODEL,
    # The T4 and P100 have no bf16 support
    "GEMMA_DTYPE": "auto" if torch.cuda.is_bf16_supported() else "float16",
    "HF_HOME": "/kaggle/working/hf",
}
print(torch.cuda.get_device_name(0), "->", env["GEMMA_DTYPE"])

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env, stdout=open("/kaggle/working/uvicorn.log", "w"), stderr=subprocess.STDOUT,
)

# The first start downloads the Gemma weights, which takes a few minutes
while True:
    if server.poll() is not None:
        print(open("/kaggle/working/uvicorn.log").read()[-3000:])
        raise RuntimeError("audio service exited")
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        break
    except OSError:
        time.sleep(10)
print("audio service is up")

In [ ]:
import re

!test -x /kaggle/working/cloudflared || (curl -sL -o /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared)
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
    stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if found:
        url = found.group(0)
        break
print(f"AUDIO_SERVICE_URL={url}" if url else open("/kaggle/working/tunnel.log").read())

Service log, for debugging a run:

In [ ]:
!tail -n 50 /kaggle/working/uvicorn.log

In [ ]:
# Follows BRANCH: when a push changes services/audio, the service restarts on the new code.
# The tunnel is not touched, so the URL stays the same. Runs until the session ends.
import json, re

SERVICE = "/kaggle/working/pitchcheck/services/audio"
LOG = "/kaggle/working/uvicorn.log"
INSTALLED = "/tmp/requirements-kaggle.txt"


def say(*words):
    print(time.strftime("%H:%M:%S UTC", time.gmtime()), *words, flush=True)


def git(*args):
    done = subprocess.run(["git", *args], cwd=SERVICE, capture_output=True, text=True, timeout=300)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr.strip()[-300:]}")
    return done.stdout.strip()


def health():
    """What /health answers, or None while the service does not answer."""
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=10) as response:
            return json.load(response)
    except (OSError, ValueError):
        return None


def busy():
    return (health() or {}).get("analysis_running", False)


def install_requirements():
    """Install requirements.txt when it differs from what was installed last."""
    # Kaggle already has CUDA builds of torch and torchvision; requirements.txt would pull CPU wheels
    wanted = "".join(line for line in open(f"{SERVICE}/requirements.txt")
                     if not re.match(r"(--extra-index-url|torch|torchvision)\b", line))
    if os.path.exists(INSTALLED) and wanted == open(INSTALLED).read():
        return
    with open(INSTALLED + ".new", "w") as file:
        file.write(wanted)
    pip = subprocess.run(["pip", "install", "-q", "-r", INSTALLED + ".new"], capture_output=True, text=True)
    if pip.returncode:
        raise RuntimeError("pip install failed: " + pip.stderr.strip()[-1000:])
    os.replace(INSTALLED + ".new", INSTALLED)
    say("installed the changed requirements.txt")


def restart():
    """Stop the service and start it on the code in the checkout. True once /health answers."""
    global server
    server.terminate()
    try:
        server.wait(timeout=60)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    server = subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=SERVICE, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    )
    deadline = time.monotonic() + 900
    while server.poll() is None and time.monotonic() < deadline:
        if health():
            return True
        time.sleep(5)
    return False


def follow():
    global running
    if busy():
        return
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
    head = git("rev-parse", "HEAD")
    if head == running:
        return
    commit = git("log", "-1", "--format=%h %s")
    # Only a change under services/audio needs a restart. When git cannot compare the two, restart.
    if subprocess.run(["git", "diff", "--quiet", running, head, "--", "."], cwd=SERVICE).returncode == 0:
        say(commit, "- nothing under services/audio changed")
        running = head
        return
    install_requirements()
    while busy():  # an analysis that started since the check above
        time.sleep(5)
    say(commit, "- restarting the service")
    started = time.monotonic()
    running = head
    if restart():
        say(f"the service is up on {commit} after {time.monotonic() - started:.0f} s")
    else:
        say("the service did not come up; the end of its log:")
        print(open(LOG).read()[-3000:], flush=True)


# The commit the service was started from
running = git("rev-parse", "HEAD")
say("following", BRANCH, "from", git("log", "-1", "--format=%h %s"))
problem = None
while True:
    time.sleep(30)
    try:
        follow()
        problem = None
    except Exception as error:  # the round is tried again in 30 s; the same error prints once
        if str(error) != problem:
            problem = str(error)
            say(problem)